In [1]:
import sys,os
sys.path.append(os.path.join(os.environ['HOME'], 'models', 'OLMT'))
import numpy as np
import pickle
import model_ELM
import matplotlib.pyplot as plt
from string import ascii_lowercase


prefix_list = ['UQ_20231118', 'UQ_20240107', 'UQ_20240112']
pft_names = ['Spruce','Tamarack','Shrub','Moss']

VAR_COL = ['GPP', 'NEE', 'HR', 'TOTVEGC', 'TOTSOMC']
VAR_PFT = ['GPP', 'AR', 'MR', 'GR', 'XR']
# variables for Xiaoying Shi
##VAR_COL = ['GPP', 'NPP', 'QVEGT', 'NEE', 'TOTVEGC']
##VAR_PFT = ['GPP', 'NPP', 'QVEGT']
pft_list = [2, 3, 11, 12]
nvars = len(VAR_COL) + len(pft_list) * len(VAR_PFT)


# break out the individual PFTs here by appending _{pft} to varname
variable_list = []
for var in VAR_COL:
    variable_list.append(var)
    if var in VAR_PFT:
        variable_list.extend([var+'_pft'+str(pft) for pft in pft_list])
for var in VAR_PFT:
    if not var in VAR_COL:
        variable_list.extend([var+'_pft'+str(pft) for pft in pft_list])


ticklabels = []
for var in VAR_COL:
    ticklabels.append(var)
    if var in VAR_PFT:
        ticklabels.extend([pname + ' ' + var for pname in pft_names])
for var in VAR_PFT:
    if not var in VAR_COL:
        ticklabels.extend([pname + ' ' + var for pname in pft_names])

x_pos = np.cumsum(np.ones(len(variable_list)))

# reorder the variable list
reorder = np.array([0, 5, 6, 7, 8, 1, 9, 13, 17, 21, 2, 10, 14, 18, 22, 
                    3, 11, 15, 19, 23, 4, 12, 16, 20, 24])

In [4]:
fig, axes = plt.subplots(3, 4, figsize = (16, 11), sharex = True, sharey = True)
fig.subplots_adjust(wspace = 0.0, hspace = 0.4)
for p, prefix in enumerate(prefix_list):
    f = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles', 
             f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC.pkl'), 'rb')
    mycase = pickle.load(f)
    f.close()

    #Plot main sensitivity indices
    for i, (pft,pftname) in enumerate(zip([2, 3, 11, 0], pft_names[:-1] + ['Column'])):
      subset = np.where(np.array(mycase.ensemble_pfts) == pft)[0]

      ax = axes[p, i]

      bottom = np.zeros(len(x_pos))
      for s in subset:
        temp = np.array([mycase.sens_main[v][s,0] for v in variable_list])
        temp = temp[reorder]
        ax.bar(x_pos, temp, align='center', # alpha=0.5,
               bottom = bottom, label = mycase.ensemble_parms[s])
        bottom = bottom + temp

      # add a line for total
      total = np.array([mycase.sens_main[v][:,0].sum() for v in variable_list])
      total = total[reorder]
      ax.plot(x_pos, total, '-k', label = 'Total')

      ax.set_xlim([x_pos[0]-0.5, x_pos[-1]+0.5])
      ax.set_xticks(x_pos)
      ax.set_xticklabels([ticklabels[t] for t in reorder], rotation=90)
      if p == 0:
        ax.set_title(f'{pftname} parameters')

      if i == 0:
        if p == 2:
            ax.legend(loc = [0.2, -0.7], ncol = 5)
        else:
            ax.legend(loc = [0.2, -0.25], ncol = 5)

      if i == 3:
        if p == 0:
            ax.legend(loc = [-0.24, -0.25], ncol = 2)
        elif p == 1:
            ax.legend(loc = [0, -0.25], ncol = 2)           
        else:
            ax.legend(loc = [0, -0.7], ncol = 2)

    for ax, lab in zip(np.ravel(axes), ascii_lowercase):
        ax.text(0, 1.05, lab, transform=ax.transAxes, fontweight = 'bold')
fig.savefig(os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 
                         'plot_ensemble_UQ_main.png'), dpi = 600.,
            bbox_inches = 'tight')

In [5]:
fig, axes = plt.subplots(3, 4, figsize = (16, 11), sharex = True, sharey = True)
fig.subplots_adjust(wspace = 0.0, hspace = 0.4)
for p, prefix in enumerate(prefix_list):
    f = open(os.path.join(os.environ['HOME'], 'models', 'OLMT', 'pklfiles', 
             f'{prefix}_US-SPR_ICB20TRCNPRDCTCBC.pkl'), 'rb')
    mycase = pickle.load(f)
    f.close()

    #Total sensitivity indices
    for i, (pft,pftname) in enumerate(zip([2, 3, 11, 0], pft_names[:-1] + ['Column'])):
      subset = np.where(np.array(mycase.ensemble_pfts) == pft)[0]

      ax = axes[p, i]

      bottom = np.zeros(len(x_pos))
      for s in subset:
        temp = np.array([mycase.sens_tot[v][s,0] for v in variable_list])
        temp = temp[reorder]
        ax.bar(x_pos, temp, align='center', # alpha=0.5,
               bottom = bottom, label = mycase.ensemble_parms[s])
        bottom = bottom + temp

      # add a line for total
      total = np.array([mycase.sens_tot[v][:,0].sum() for v in variable_list])
      total = total[reorder]
      ax.plot(x_pos, total, '-k', label = 'Total')

      ax.set_xlim([x_pos[0]-0.5, x_pos[-1]+0.5])
      ax.set_xticks(x_pos)
      ax.set_xticklabels([ticklabels[t] for t in reorder], rotation=90)
      ax.set_title(f'{pftname} parameters')

      if i == 0:
        if p == 2:
            ax.legend(loc = [0.2, -0.7], ncol = 5)
        else:
            ax.legend(loc = [0.2, -0.25], ncol = 5)

      if i == 3:
        if p == 0:
            ax.legend(loc = [-0.24, -0.25], ncol = 2)
        elif p == 1:
            ax.legend(loc = [0, -0.25], ncol = 2)           
        else:
            ax.legend(loc = [0, -0.7], ncol = 2)

    for ax, lab in zip(np.ravel(axes), ascii_lowercase):
        ax.text(-0.05, 1.05, lab, transform=ax.transAxes, fontweight = 'bold')
fig.savefig(os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 
                         'plot_ensemble_UQ_tot.png'), dpi = 600.,
            bbox_inches = 'tight')